# Aria ML/AI Development Guide

**Complete guide for machine learning, LoRA fine-tuning, model evaluation, and autonomous training workflows.**

Learn how to train, evaluate, and deploy AI models in Aria.


## ML Development Architecture

### Training Pipeline Overview

```
Data Collection
    ↓
Dataset Validation
    ↓
LoRA Fine-Tuning
    ├─ Base Model (Phi-3.5, TinyLlama)
    ├─ Dataset (JSONL format)
    ├─ Hyperparameters (epochs, lr, batch_size)
    └─ Training Script
    ↓
Model Evaluation
    ├─ Metrics (accuracy, perplexity, diversity)
    ├─ Comparison (baseline vs fine-tuned)
    └─ Quality Assessment
    ↓
Model Promotion
    ├─ Staging (shadow deployment)
    ├─ Monitoring (A/B testing)
    └─ Production (live users)
    ↓
Continuous Improvement
    ├─ Collect inference data
    ├─ Retrain periodically
    └─ Auto-promote on improvement
```

### Project Structure

```
ai-projects/
├─ lora-training/
│  ├─ microsoft_phi-silica-3.6_v1/
│  │  ├─ scripts/
│  │  │  ├─ train.py           # Main training script
│  │  │  ├─ evaluate.py        # Model evaluation
│  │  │  ├─ benchmark.py       # Performance testing
│  │  │  └─ deploy.py          # Model promotion
│  │  ├─ config/
│  │  │  ├─ train_config.yaml  # Training hyperparameters
│  │  │  └─ eval_config.yaml   # Evaluation config
│  │  ├─ checkpoints/          # Saved models
│  │  └─ requirements.txt      # Dependencies
│
├─ quantum-ml/
│  ├─ src/
│  │  ├─ quantum_llm/          # Quantum LLM integration
│  │  ├─ quantum_sampler.py    # Token sampling with quantum
│  │  └─ quantum_embeddings.py # Quantum embeddings
│  └─ quantum_mcp_server.py    # MCP server for quantum tools
│
└─ chat-cli/
   └─ src/
      ├─ chat_providers.py     # Provider detection
      ├─ token_utils.py        # Token counting
      ├─ agi_provider.py       # AGI reasoning
      └─ lora_inference.py     # LoRA inference bridge
```

---

## LoRA Fine-Tuning

### LoRA Concepts

```
Base Model (frozen)
    ↓
LoRA Adapter (trainable, ~0.5% of model size)
├─ Rank: r (typically 8, 16, 32)
├─ Alpha: α (scaling factor, typically 16-32)
├─ Modules: attention, feedforward
└─ Output: Linear combination of adapters
    ↓
Inference: Base + LoRA adapter
Result: Improved performance with minimal parameters
```

### Training Script Example

```python
# scripts/train_lora_adapter.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# Configuration
BASE_MODEL = "microsoft/phi-2"  # or "TinyLlama/TinyLlama-1.1B"
LORA_RANK = 16
LORA_ALPHA = 32
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 2e-4

def setup_lora_model():
    """Load base model and setup LoRA."""
    # Load base model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    # Configure LoRA
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj"],  # Which modules to fine-tune
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA to model
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Output: trainable params: 1,720,320 || all params: 2,610,642,944 || trainable%: 0.07

    return model

def train_adapter(model, dataset_path):
    """Train LoRA adapter."""
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    # Load dataset
    dataset = load_dataset("json", data_files=dataset_path)

    # Tokenize
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=512,
            padding="max_length"
        )

    tokenized = dataset.map(tokenize_function, batched=True)

    # Training arguments
    training_args = TrainingArguments(
        output_dir="./checkpoints",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_steps=100,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=100,
        eval_steps=50,
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"] if "validation" in tokenized else None,
        data_collator=lambda data: {
            k: torch.stack([f[k] for f in data])
            for k in data[0].keys()
        }
    )

    # Train
    trainer.train()

    # Save adapter
    model.save_pretrained("./adapter_output")
    print("✓ LoRA adapter saved to ./adapter_output")

if __name__ == "__main__":
    model = setup_lora_model()
    train_adapter(model, "datasets/chat/training_data.jsonl")
```

### Training Dataset Format

```jsonl
{"text": "User: What is machine learning?\nAssistant: Machine learning is..."}
{"text": "User: How do I fine-tune a model?\nAssistant: To fine-tune..."}
{"text": "User: What is LoRA?\nAssistant: LoRA (Low-Rank Adaptation)..."}
```

**Dataset Validation:**

```python
# scripts/validate_dataset.py
import json
from pathlib import Path

def validate_training_data(dataset_path: str):
    """Validate training dataset format."""
    issues = []

    with open(dataset_path) as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)

                # Check required fields
                if "text" not in obj:
                    issues.append(f"Line {i}: Missing 'text' field")

                # Check text length
                text = obj["text"]
                if len(text) < 10:
                    issues.append(f"Line {i}: Text too short (<10 chars)")
                if len(text) > 10000:
                    issues.append(f"Line {i}: Text too long (>10000 chars)")

            except json.JSONDecodeError:
                issues.append(f"Line {i}: Invalid JSON")

    if issues:
        print(f"❌ Found {len(issues)} issues:")
        for issue in issues[:10]:  # Show first 10
            print(f"  - {issue}")
    else:
        print(f"✓ Dataset valid ({i} lines, {sum(1 for _ in open(dataset_path))} total)")

if __name__ == "__main__":
    validate_training_data("datasets/chat/training_data.jsonl")
```

---

## Model Evaluation

### Evaluation Metrics

```python
# scripts/evaluate_model.py
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import numpy as np

def evaluate_model(model_path: str, test_dataset_path: str):
    """Evaluate model performance."""
    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Metric 1: Perplexity (lower is better)
    print("📊 Computing perplexity...")
    perplexities = []
    with open(test_dataset_path) as f:
        for line in f:
            text = json.loads(line)["text"]
            inputs = tokenizer(text, return_tensors="pt")
            with torch.no_grad():
                outputs = model(**inputs, labels=inputs["input_ids"])
                loss = outputs.loss
                perplexity = torch.exp(loss)
            perplexities.append(perplexity.item())

    mean_perplexity = np.mean(perplexities)
    print(f"  Perplexity: {mean_perplexity:.2f} (lower is better)")

    # Metric 2: Accuracy on test set
    print("📊 Computing accuracy...")
    correct = 0
    total = 0
    with open(test_dataset_path) as f:
        for line in f:
            text = json.loads(line)["text"]
            # Simulate accuracy test
            inputs = tokenizer(text[:100], return_tensors="pt")
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits
            # Check if prediction matches expected
            # (simplified for example)
            total += 1
            correct += 1

    accuracy = correct / total if total > 0 else 0
    print(f"  Accuracy: {accuracy:.2%}")

    # Metric 3: Diversity (different responses for similar inputs)
    print("📊 Computing diversity...")
    diversities = []
    similar_prompts = [
        "What is AI?",
        "What is artificial intelligence?",
        "Tell me about AI"
    ]
    responses = []
    for prompt in similar_prompts:
        inputs = tokenizer(prompt, return_tensors="pt")
        output = model.generate(**inputs, max_length=50, temperature=0.7)
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        responses.append(response)

    # Compute pairwise diversity (using simple string similarity)
    diversity = compute_diversity(responses)
    print(f"  Diversity: {diversity:.2f}/100 (higher is better)")

    return {
        "perplexity": mean_perplexity,
        "accuracy": accuracy,
        "diversity": diversity
    }

def compute_diversity(texts: list[str]) -> float:
    """Compute diversity between texts (0-100)."""
    from difflib import SequenceMatcher
    similarities = []
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            ratio = SequenceMatcher(None, texts[i], texts[j]).ratio()
            similarities.append(ratio)

    avg_similarity = np.mean(similarities) if similarities else 0
    return (1 - avg_similarity) * 100
```

### Batch Evaluation

```bash
# Evaluate multiple models
python -m scripts.batch_evaluator \
  --models checkpoints/model_v1,checkpoints/model_v2 \
  --test-dataset datasets/test/evaluation.jsonl \
  --output data_out/evaluation_results.json

# Output:
# {
#   "model_v1": {"perplexity": 45.2, "accuracy": 0.92, "diversity": 75.3},
#   "model_v2": {"perplexity": 42.1, "accuracy": 0.94, "diversity": 78.1}
# }
```

---

## Inference & Integration

### Using LoRA Adapter in Chat

```python
# shared/lora_inference_bridge.py
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

class LoRAInferenceBridge:
    def __init__(self, base_model_name: str, adapter_path: str):
        """Load base model and LoRA adapter."""
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        self.model = AutoModelForCausalLM.from_pretrained(base_model_name)

        # Load LoRA adapter
        self.model = PeftModel.from_pretrained(self.model, adapter_path)
        self.model.eval()

    def generate_response(self, prompt: str, max_tokens: int = 100) -> str:
        """Generate response using LoRA-enhanced model."""
        inputs = self.tokenizer(prompt, return_tensors="pt")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

# Usage in chat provider
lora_bridge = LoRAInferenceBridge(
    base_model_name="microsoft/phi-2",
    adapter_path="checkpoints/chat_adapter"
)

def chat_with_lora(message: str) -> str:
    """Chat using LoRA-enhanced model."""
    return lora_bridge.generate_response(message)
```

---

## Autonomous Training

### Continuous Improvement Cycle

```python
# scripts/autonomous_training_orchestrator.py
import asyncio
from datetime import datetime
import json

class AutonomousTrainingOrchestrator:
    def __init__(self, config_path: str):
        """Initialize autonomous training."""
        with open(config_path) as f:
            self.config = json.load(f)

        self.cycle_count = 0
        self.best_accuracy = 0

    async def run_training_cycle(self):
        """Run one complete training cycle."""
        print(f"\n=== Training Cycle {self.cycle_count} ===")

        # Step 1: Discover datasets
        print("[1/5] Discovering datasets...")
        datasets = await self.discover_datasets()

        # Step 2: Select hyperparameters
        print("[2/5] Selecting hyperparameters...")
        epochs = self.select_optimal_epochs()

        # Step 3: Train model
        print("[3/5] Training model...")
        model_path = await self.train_model(datasets, epochs)

        # Step 4: Evaluate
        print("[4/5] Evaluating model...")
        metrics = await self.evaluate_model(model_path)

        # Step 5: Promote if improved
        print("[5/5] Checking for promotion...")
        if metrics["accuracy"] > self.best_accuracy * 1.05:  # 5% improvement
            await self.promote_model(model_path, metrics)
            self.best_accuracy = metrics["accuracy"]

        self.cycle_count += 1
        return metrics

    async def discover_datasets(self) -> list[str]:
        """Discover available training datasets."""
        datasets = []
        for source in self.config["dataset_sources"]:
            found = await self.scan_source(source)
            datasets.extend(found)
        return datasets[:self.config["max_datasets"]]

    def select_optimal_epochs(self) -> int:
        """Select epochs based on performance history."""
        if self.best_accuracy < 0.7:
            return 100  # Increase training for low accuracy
        elif self.best_accuracy < 0.85:
            return 50
        else:
            return 25  # Reduce training for high accuracy

    async def run_forever(self):
        """Run training cycles indefinitely."""
        while True:
            try:
                await self.run_training_cycle()
                await asyncio.sleep(self.config["cycle_interval"])
            except Exception as e:
                print(f"❌ Error in training cycle: {e}")
                await asyncio.sleep(60)
```


## ML Best Practices Checklist

### Dataset Management

- [ ] Data is JSONL format with "text" field
- [ ] No duplicate entries
- [ ] Text length 10-10000 characters
- [ ] Dataset validated before training
- [ ] Datasets are read-only after creation
- [ ] Version control for datasets
- [ ] README with dataset description

### Training

- [ ] Hyperparameters documented
- [ ] Training metrics logged
- [ ] Checkpoints saved regularly
- [ ] Memory usage monitored
- [ ] Training time tracked
- [ ] Early stopping implemented
- [ ] Reproducible (fixed random seed)

### Evaluation

- [ ] Test set separate from training
- [ ] Multiple metrics computed
- [ ] Baseline comparison done
- [ ] Error analysis performed
- [ ] Results documented
- [ ] Evaluation reproducible
- [ ] A/B test before promotion

### Production

- [ ] Model size < deployment limit
- [ ] Inference latency acceptable
- [ ] Edge cases tested
- [ ] Graceful degradation implemented
- [ ] Rollback plan documented
- [ ] Monitoring alerts configured
- [ ] Version tracking enabled

### ML Operations (MLOps)

- [ ] Models tracked in registry
- [ ] Training pipeline automated
- [ ] Evaluation automated
- [ ] Promotion criteria defined
- [ ] Retraining schedule set
- [ ] Performance degradation detected
- [ ] Continuous monitoring
